# 📊 Analytica — Asistente experto en Google Analytics 4
### Proyecto Final · IA Generativa — Gemini + RAG + LangGraph

Agente conversacional que responde preguntas sobre **Google Analytics 4 (GA4)** usando como base de conocimiento la documentación oficial de Google indexada en **ChromaDB** con **Gemini Embeddings**. El agente está construido con **LangGraph**, usa **Gemini** como LLM y tiene **memoria de conversación**.

**Índice**
1. Configuración del entorno
2. Definición del dominio
3. Base de conocimiento (carga → limpieza → chunking → ChromaDB)
4. System prompt
5. Agente en LangGraph
6. Memoria de conversación
7. Preguntas de ejemplo documentadas
8. Chat interactivo

## 1. Configuración del entorno
Funciona en **Google Colab** y en **Jupyter local**. En Colab, la celda clona el repositorio e instala las dependencias.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/joseaespinosa02/asistente-ga4-rag.git"

if IN_COLAB:
    %cd /content
    if not os.path.exists("asistente-ga4-rag"):
        !git clone -q {REPO_URL}
    %cd /content/asistente-ga4-rag
    !git pull -q
    !pip install -q -r requirements.txt
else:
    # En local el notebook vive en /notebooks: subimos a la raíz del proyecto
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Directorio de trabajo:", os.getcwd())

### API key de Gemini
Nunca se escribe en el código. Se lee de:
- **Colab:** icono de la llave 🔑 (Secrets) → `GOOGLE_API_KEY`
- **Local:** archivo `.env` en la raíz (ver `.env.example`)

In [ ]:
from src import config

config.load_api_key()
print("✓ API key cargada")
print("LLM:", config.LLM_MODEL, "| Embeddings:", config.EMBEDDING_MODEL)

## 2. Definición del dominio

**Dominio:** analítica web con **Google Analytics 4**.

**Por qué este tema:** GA4 es la herramienta estándar de medición web, pero su documentación está repartida en decenas de artículos y cambia a menudo (por ejemplo, «conversiones» pasó a llamarse «eventos clave»). Un asistente que responda con la documentación oficial y cite la fuente resuelve dudas reales de equipos de marketing y growth en segundos.

**Base de conocimiento:** 16 artículos oficiales en español (Centro de Ayuda de Google Analytics + guías para desarrolladores) que cubren configuración, eventos, parámetros, dimensiones personalizadas, depuración, canales y atribución. Además se pueden añadir PDFs propios en la carpeta `data/`.

In [ ]:
import pandas as pd

pd.DataFrame(config.SOURCES).rename(columns={"title": "Documento", "url": "URL"})

## 3. Base de conocimiento vectorial

### 3.1 Carga y limpieza
Cada página se descarga, se limpia con BeautifulSoup (quitando menús, scripts, pies de página y textos de interfaz como «¿Te ha resultado útil?») y se guarda en caché en `data/cache/` para no descargarla de nuevo.

In [ ]:
from src.ingest import load_documents, split_documents

docs = load_documents()

resumen = pd.DataFrame(
    [{"Documento": d.metadata["title"], "Caracteres": len(d.page_content)} for d in docs]
)
resumen["Páginas aprox."] = (resumen["Caracteres"] / 3000).round(1)  # ~3.000 caracteres por página
display(resumen)
print(f"\n{len(docs)} documentos · {resumen['Caracteres'].sum():,} caracteres · ~{resumen['Páginas aprox.'].sum():.0f} páginas")

In [ ]:
# Vista previa del texto limpio de un documento
print(docs[0].page_content[:1500])

### 3.2 Chunking
Se usa `RecursiveCharacterTextSplitter` con **1.000 caracteres** por fragmento y **150 de solapamiento**:
- 1.000 caracteres ≈ 1-2 párrafos: suficiente para que un fragmento contenga una idea completa (por ejemplo, la definición de un evento y sus parámetros) sin diluir el embedding con temas mezclados.
- El solapamiento evita perder información que queda justo en el corte entre dos fragmentos.
- Los separadores priorizan cortar por párrafo → línea → frase, para no partir oraciones.

In [ ]:
chunks = split_documents(docs)
largos = pd.Series([len(c.page_content) for c in chunks])
print(f"{len(chunks)} fragmentos · longitud media {largos.mean():.0f} caracteres (mín {largos.min()}, máx {largos.max()})")
print("\nEjemplo de fragmento:\n")
print(chunks[10].metadata)
print(chunks[10].page_content)

### 3.3 Embeddings con Gemini e indexación en ChromaDB
Cada fragmento se convierte en un vector con `gemini-embedding-001` y se guarda en una colección persistente de ChromaDB (`chroma_db/`) con distancia coseno. La indexación va por lotes y reintenta automáticamente si la capa gratuita de la API devuelve un error de límite (429).

In [ ]:
from src.ingest import build_vectorstore, get_embeddings

embeddings = get_embeddings()
vectorstore = build_vectorstore(chunks, embeddings)
print("\n✓ Fragmentos en ChromaDB:", vectorstore._collection.count())

### 3.4 Verificación de la colección antes de conectar el agente
Consultas de prueba directas a ChromaDB para comprobar que la recuperación devuelve los documentos correctos (distancia coseno: cuanto más baja, más parecido).

In [ ]:
consultas_prueba = [
    "¿Qué eventos se recogen con la medición mejorada?",
    "¿Se puede editar el grupo de canales predeterminado?",
    "¿Qué parámetros lleva el evento purchase?",
    "¿Cómo depuro eventos en tiempo real?",
]

for q in consultas_prueba:
    print(f"\n🔎 {q}")
    for doc, dist in vectorstore.similarity_search_with_score(q, k=3):
        print(f"   {dist:.3f} · {doc.metadata['title']} · «{doc.page_content[:90]}…»")

## 4. System prompt

El prompt completo está en `src/agent.py` y su justificación en el README. Resumen de decisiones:

| Decisión | Motivo |
|---|---|
| Rol de consultor GA4 con nombre («Analytica») | Fija el dominio y un tono coherente durante toda la conversación |
| Responder **solo** con el contexto recuperado | Reduce alucinaciones: en GA4 es fácil inventar nombres de eventos o rutas de menú |
| Frase explícita cuando no hay información | Mejor admitir el límite que inventar |
| Usar el historial para resolver referencias («eso», «el segundo») | Coherencia conversacional junto con la memoria |
| Citar fuentes con su título | Permite verificar la respuesta en la documentación oficial |
| Respuesta directa primero, luego pasos numerados, máximo ~250 palabras | Formato útil para un profesional que está configurando GA4 |
| Rechazo amable de temas fuera de GA4 y sin asesoría legal | Delimita el alcance y evita consejos de privacidad que requieren un especialista |

In [ ]:
from src.agent import SYSTEM_PROMPT, CONTEXTUALIZE_PROMPT

print(SYSTEM_PROMPT)
print("=" * 80)
print("Prompt de reformulación (nodo contextualize):\n")
print(CONTEXTUALIZE_PROMPT)

## 5. Agente en LangGraph

El grafo tiene tres nodos que se ejecutan en **cada** pregunta, de modo que el agente siempre consulta la base de conocimiento:

1. **`contextualize`**: si hay historial, Gemini reescribe la pregunta para que se entienda sola (*«¿y cómo se desactiva?»* → *«¿cómo se desactivan los eventos de medición mejorada en GA4?»*). Sin este paso, el retriever buscaría con una pregunta sin contexto y la memoria no serviría para el RAG.
2. **`retrieve`**: busca en ChromaDB los 5 fragmentos más relevantes con **MMR** (Maximal Marginal Relevance), que equilibra relevancia y diversidad para no pasar 5 fragmentos casi idénticos.
3. **`generate`**: Gemini responde con el system prompt + contexto recuperado + historial reciente.

In [ ]:
from src.agent import build_agent, ask

agent = build_agent(vectorstore)

try:
    from IPython.display import Image
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print(agent.get_graph().draw_mermaid())

## 6. Memoria de conversación

La memoria la gestiona el **checkpointer** de LangGraph (`InMemorySaver`): cada conversación se identifica con un `thread_id` y el estado completo (mensajes, pregunta reformulada y contexto) se guarda tras cada paso. Al invocar de nuevo el agente con el mismo `thread_id`, recupera el historial automáticamente. Con otro `thread_id` empieza de cero.

In [ ]:
def chat(pregunta: str, thread_id: str = "demo", mostrar_detalle: bool = True):
    """Envía una pregunta al agente y muestra la respuesta con su trazabilidad."""
    estado = ask(agent, pregunta, thread_id=thread_id)
    print(f"🧑 {pregunta}")
    if mostrar_detalle and estado["standalone_question"] != pregunta:
        print(f"   ↳ búsqueda reformulada: «{estado['standalone_question']}»")
    print(f"\n🤖 {estado['messages'][-1].text}")
    if mostrar_detalle:
        fuentes = sorted({c["title"] for c in estado["context"]})
        print(f"\n   📚 Fragmentos de: {', '.join(fuentes)}")
    print("\n" + "─" * 100 + "\n")
    return estado

## 7. Preguntas de ejemplo documentadas

Todas en el mismo hilo (`thread_id="demo"`) para demostrar la memoria. Las preguntas **3** y **6** solo se pueden responder bien recordando lo anterior.

**Pregunta 1 — Conceptos básicos:** tipos de eventos.

In [ ]:
_ = chat("¿Qué tipos de eventos existen en GA4 y en qué se diferencian?")

**Pregunta 2 — Detalle técnico:** medición mejorada.

In [ ]:
_ = chat("¿Qué eventos recoge la medición mejorada?")

**Pregunta 3 — 🧠 Memoria:** «la» se refiere a la medición mejorada de la pregunta anterior.

In [ ]:
_ = chat("¿Y cómo la activo o desactivo?")

**Pregunta 4 — Implementación:** comercio electrónico.

In [ ]:
_ = chat("¿Qué parámetros necesita el evento purchase para medir ventas?")

**Pregunta 5 — Adquisición:** grupo de canales.

In [ ]:
_ = chat("¿Puedo editar el grupo de canales predeterminado? Si no, ¿qué alternativa tengo?")

**Pregunta 6 — 🧠 Memoria:** pregunta sobre la conversación misma.

In [ ]:
_ = chat("Resúmeme en 3 puntos lo que hemos visto sobre eventos al principio de la conversación.")

**Pregunta 7 — Límites del agente:** tema fuera del dominio.

In [ ]:
_ = chat("¿Me recomiendas una receta de paella?")

### Comprobación de la memoria
El estado guardado por el checkpointer contiene toda la conversación del hilo `demo`. Un hilo nuevo no recuerda nada.

In [ ]:
estado = agent.get_state({"configurable": {"thread_id": "demo"}})
print(f"Mensajes guardados en el hilo 'demo': {len(estado.values['messages'])}")

_ = chat("¿Cuál fue mi primera pregunta?", thread_id="hilo-nuevo")

## 8. Chat interactivo
Escribe tus preguntas; `salir` para terminar. Usa un hilo propio, así que empieza sin memoria previa.

In [ ]:
import uuid

hilo = f"interactivo-{uuid.uuid4().hex[:6]}"
print("Chat con Analytica (escribe 'salir' para terminar)\n")
while True:
    pregunta = input("🧑 ")
    if pregunta.strip().lower() in {"salir", "exit", "quit", ""}:
        print("¡Hasta luego!")
        break
    chat(pregunta, thread_id=hilo, mostrar_detalle=False)

## Conclusiones

- **RAG con Gemini + ChromaDB:** la documentación oficial de GA4 (16 artículos, más de 20 páginas) se indexa con `gemini-embedding-001` y se recupera con MMR antes de cada respuesta.
- **Agente en LangGraph:** el flujo `contextualize → retrieve → generate` garantiza que cada respuesta se apoya en la base de conocimiento, incluso en preguntas de seguimiento.
- **Memoria:** el checkpointer de LangGraph mantiene el historial por `thread_id`, y el nodo de reformulación hace que esa memoria también mejore la búsqueda.
- **Prompt:** el agente cita fuentes, admite cuando no sabe algo y rechaza temas fuera de su dominio.

**Mejoras posibles:** re-ranking de los fragmentos, búsqueda híbrida (BM25 + vectores), evaluación automática con un conjunto de preguntas de referencia y actualización periódica de la documentación.